# 1️⃣ Flood Detection — UNet++ with MNDWI Band Engineering

**What this does:**
Computes the MNDWI index (Modified Normalized Difference Water Index) on-the-fly and adds it as a 7th channel to the EfficientNet. Saves predictions to `smp_predictions/`.




## Cell 1 — Install & Imports




In [ ]:
!pip install -q segmentation-models-pytorch albumentations rasterio

import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from glob import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import rasterio



## Cell 2 — Config & Paths




In [ ]:
DATA_ROOT = '/kaggle/input/anrfaisehack-theme-1-phase2/data/'
OUT_DIR = '/kaggle/working/'
# We will save these TIFs to a dedicated folder for ensembling later
PREDICT_SMP_DIR = os.path.join(OUT_DIR, "smp_predictions")
os.makedirs(PREDICT_SMP_DIR, exist_ok=True)

EPOCHS = 80
BATCH_SIZE = 8
IN_CHANNELS = 7 # 6 Original + 1 MNDWI

MEANS = [841.1257, 371.6175, 1734.1789, 1588.3142, 1742.8452, 1218.5551]
STDS  = [473.7090, 170.3611, 623.0474, 612.8465, 564.5835, 528.0894]



## Cell 3 — Dataset with MNDWI Band Engineering




In [ ]:
class FloodDataset(Dataset):
    def __init__(self, split_file, transform=None, is_train=False, is_predict=False):
        self.transform = transform
        self.is_train = is_train
        self.is_predict = is_predict
        img_dir = DATA_ROOT + 'image/'
        lbl_dir = DATA_ROOT + 'label/'
        
        if split_file and os.path.exists(split_file):
            with open(split_file) as f: ids = [l.strip() for l in f if l.strip()]
            self.imgs = [os.path.join(img_dir, f"{i}_image.tif") for i in ids]
            if not is_predict:
                self.lbls = [os.path.join(lbl_dir, f"{i}_label.tif") for i in ids]
        else:
            self.imgs = sorted(glob(img_dir + "*.tif"))
            if not is_predict: self.lbls = sorted(glob(lbl_dir + "*.tif"))

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        with rasterio.open(self.imgs[idx]) as src:
            raw = src.read().astype(np.float32)
            
        # 🌟 INNOVATION: Compute MNDWI before normalisation
        # MNDWI = (Green - SWIR) / (Green + SWIR). Green is index 2, SWIR is index 5
        mndwi = (raw[2] - raw[5]) / (raw[2] + raw[5] + 1e-8)
        
        # Normalize the 6 original bands
        img = raw.copy()
        for b in range(6): img[b] = (img[b] - MEANS[b]) / STDS[b]
        
        # Stack meaning 7 channels
        img = np.nan_to_num(img, nan=0.0)
        mndwi = np.nan_to_num(mndwi, nan=0.0)
        img = np.concatenate([img, np.expand_dims(mndwi, axis=0)], axis=0) # [7, H, W]

        # Modal Dropout (Only in training)
        if self.is_train:
            r = random.random()
            if r < 0.20: img[2:6, :, :] = 0.0  # Drop optical
            elif r < 0.30: img[0:2, :, :] = 0.0 # Drop SAR

        img = img.transpose(1, 2, 0) # HWC

        if self.is_predict:
            if self.transform: img = self.transform(image=img)['image']
            else: img = torch.from_numpy(img.transpose(2,0,1))
            return img, self.imgs[idx]

        with rasterio.open(self.lbls[idx]) as src: mask = src.read(1).astype(np.int64)
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        else:
            img, mask = torch.from_numpy(img.transpose(2,0,1)), torch.from_numpy(mask)

        return img, mask

t_trans = A.Compose([A.D4(), A.RandomCrop(512, 512), ToTensorV2()])
v_trans = A.Compose([ToTensorV2()])

train_ds = FloodDataset(DATA_ROOT + 'split/train.txt', t_trans, is_train=True)
val_ds   = FloodDataset(DATA_ROOT + 'split/val.txt', v_trans)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)



## Cell 4 — Model & Loss




In [ ]:
model = smp.UnetPlusPlus('efficientnet-b4', encoder_weights="imagenet", in_channels=7, classes=3).cuda()

class KaggleWinnerLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.lovasz = smp.losses.LovaszLoss('multiclass', ignore_index=-1)
        self.focal = smp.losses.FocalLoss('multiclass', ignore_index=-1, gamma=2.0)
    def forward(self, logits, masks):
        return 0.7 * self.lovasz(logits, masks) + 0.3 * self.focal(logits, masks)

criterion = KaggleWinnerLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1.5e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')



## Cell 5 — Train Loop




In [ ]:
best_m = 0.0
print("Training SMP 7-Channel...")
for epoch in range(EPOCHS):
    model.train()
    for imgs, masks in train_loader:
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model(imgs.cuda()), masks.cuda())
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    scheduler.step()
    
    # Save best roughly (simplified logic for speed)
    model.eval()
    t_loss = loss.item()
    if epoch % 10 == 0 or epoch == EPOCHS - 1:
        torch.save(model.state_dict(), f"{OUT_DIR}/smp_mndwi.pth")
        print(f"Epoch {epoch} finished.")



## Cell 6 — Predict & Save TIFs (with TTA)




In [ ]:
model.load_state_dict(torch.load(f"{OUT_DIR}/smp_mndwi.pth", weights_only=True))
model.eval()

predict_files = sorted(Path(DATA_ROOT + 'prediction/image/').glob("*.tif"))
for p in predict_files:
    with rasterio.open(p) as src:
        raw = src.read().astype(np.float32)
        meta = src.meta.copy()
    
    mndwi = (raw[2] - raw[5]) / (raw[2] + raw[5] + 1e-8)
    for b in range(6): raw[b] = (raw[b] - MEANS[b]) / STDS[b]
    raw = np.nan_to_num(raw, nan=0.0)
    img = np.concatenate([raw, np.expand_dims(mndwi, axis=0)], axis=0) 
    
    x = torch.from_numpy(img).unsqueeze(0).float().cuda()
    with torch.no_grad(), torch.amp.autocast('cuda'):
        # TTA
        p1 = F.softmax(model(x), dim=1)
        p2 = torch.flip(F.softmax(model(torch.flip(x, dims=[3])), dim=1), dims=[3])
        p3 = torch.flip(F.softmax(model(torch.flip(x, dims=[2])), dim=1), dims=[2])
        probs = ((p1 + p2 + p3) / 3.0).squeeze(0).cpu().numpy()
        
    # We save purely the argmax for the ensemble
    pred = np.argmax(probs, axis=0).astype("int16")
    
    meta.update({"count": 1, "dtype": "int16", "nodata": -1})
    with rasterio.open(os.path.join(PREDICT_SMP_DIR, p.name), "w", **meta) as dst:
        dst.write(pred, 1)

print("✅ Saved SMP Predictions!")
